In [10]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from plot_func import error_scatter, interval_score, coverage, performance_dist, table

# Test Case 1 experimental data
# folder = "interp_reg_temp"

folder = "synthetic_data/case11/pyvale-output/interp_reg_temp"

metrics = ["total_plus", "total_minus"]
models = [
    "1", "2", "3", "4"
]

TOLERANCE=0.1

In [11]:

for metric in metrics:
    
    dfs = []
    for model in models:
                df = pd.read_csv(f"../../{folder}/ablation_results/{metric}_ablation_{model}.csv")
                df["model"] = model
                dfs.append(df)

    df_all = pd.concat(dfs, ignore_index=True)
    
    fig, ax = error_scatter(df_all, TOLERANCE, tag=metric)
    save_path = f"../../{folder}/plots/error_scatter_{metric}_{model}.jpg"
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    fig, ax = interval_score(df_all, models, tag=metric)
    save_path = f"../../{folder}/plots/interval_score_{metric}_{model}.jpg"
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    fig, ax = coverage(df_all, tag=metric)
    save_path = f"../../{folder}/plots/coverage_{metric}_{model}.jpg"
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.


/home/wiera/Documents/fullfieldvalmetrics/scripts/plots/plot_func.py:101: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_all.groupby("model")["within_pi"]
Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.
/home/wiera/Documents/fullfieldvalmetrics/scripts/plots/plot_func.py:101: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_all.groupby("model")["within_pi"]


In [12]:
all_results = []

for metric in metrics:

    for model in models:

        # filepath = (f"../../{folder}/ablation_results/"
        #             f"{metric}_ablation_{model_type_print}.csv"
        # )
        filepath = f"../../{folder}/ablation_summary_{model}.csv"

        df = pd.read_csv(filepath)

        # Store the model information with each result
        df["metric"] = metric
        df["model_type"] = model


        all_results.append(df)


# Combine all summary files
results = pd.concat(all_results, ignore_index=True)
print(results)

              d_type       MAE      RMSE          MAPE  mean_abs_error  \
0           sim_plus  2.428734  2.815194    318.808204        2.428734   
1          sim_minus  2.286065  2.644162    326.313215        2.286065   
2    model_form_plus  2.297161  2.571142  10699.476051        2.297161   
3   model_form_minus  0.783333  0.973008     10.566611        0.783333   
4         total_plus  4.608772  5.278266     67.075496        4.608772   
5        total_minus  2.980591  3.572519     26.538189        2.980591   
6           sim_plus  1.872929  2.713877    264.037228        1.872929   
7          sim_minus  1.775535  2.568368    300.969064        1.775535   
8    model_form_plus  2.190727  3.353621    754.584682        2.190727   
9   model_form_minus  0.345320  0.552817      3.727398        0.345320   
10        total_plus  3.783673  5.984127     38.352027        3.783673   
11       total_minus  2.073868  3.063866     13.790012        2.073868   
12          sim_plus  2.428734  2.8151

In [13]:
best_mean_rel_error = (
    results.loc[
        results.groupby("d_type")["mean_rel_error"].idxmin()
    ]
)

print(best_mean_rel_error[
    ["d_type", "model_type", "mean_rel_error", "pi_coverage", "mean_interval_score"]
])

df_to_plot = best_mean_rel_error[
    ["d_type", "model_type", "mean_rel_error", "pi_coverage", "mean_interval_score"]
]

fig, ax = table(df_to_plot)
save_path = f"../../{folder}/plots/best_mean_rel_error.jpg"
fig.savefig(save_path, dpi=300, bbox_inches="tight")
plt.close(fig)

              d_type model_type  mean_rel_error  pi_coverage  \
9   model_form_minus          2        0.037274     0.714286   
8    model_form_plus          2        7.545847     0.857143   
7          sim_minus          2        3.009691     1.000000   
6           sim_plus          2        2.640372     1.000000   
11       total_minus          2        0.137900     0.857143   
22        total_plus          4        0.364429     0.714286   

    mean_interval_score  
9              2.840257  
8             16.348381  
7              8.970338  
6              9.407707  
11             9.791029  
22            48.512506  


In [14]:
best_interval_score = (
    results.loc[
        results.groupby("d_type")["mean_interval_score"].idxmin()
    ]
)

print(best_interval_score[
    ["d_type", "model_type", "mean_rel_error", "pi_coverage", "mean_interval_score"]
])

df_to_plot = best_interval_score[
    ["d_type", "model_type", "mean_rel_error", "pi_coverage", "mean_interval_score"]
]

fig, ax = table(df_to_plot)
save_path = f"../../{folder}/plots/best_interval_score.jpg"
fig.savefig(save_path, dpi=300, bbox_inches="tight")
plt.close(fig)

              d_type model_type  mean_rel_error  pi_coverage  \
9   model_form_minus          2        0.037274     0.714286   
14   model_form_plus          3      121.827666     0.857143   
13         sim_minus          3        3.263132     1.000000   
12          sim_plus          3        3.188082     1.000000   
11       total_minus          2        0.137900     0.857143   
16        total_plus          3        0.683766     1.000000   

    mean_interval_score  
9              2.840257  
14            12.069840  
13             8.832217  
12             9.384516  
11             9.791029  
16            18.461285  


In [15]:
target_coverage = 0.95

results["coverage_distance"] = (
    results["pi_coverage"] - target_coverage
).abs()

best_coverage = (
    results.loc[
        results.groupby("d_type")["coverage_distance"].idxmin()
    ]
)

print(best_coverage[
    ["d_type", "model_type", "mean_rel_error", "pi_coverage", "mean_interval_score"]
])

df_to_plot = best_coverage[
    ["d_type", "model_type", "mean_rel_error", "pi_coverage", "mean_interval_score"]
]

fig, ax = table(df_to_plot)
save_path = f"../../{folder}/plots/best_coverage.jpg"
fig.savefig(save_path, dpi=300, bbox_inches="tight")
plt.close(fig)


             d_type model_type  mean_rel_error  pi_coverage  \
3  model_form_minus          1        0.105666          1.0   
2   model_form_plus          1      106.994761          1.0   
1         sim_minus          1        3.263132          1.0   
0          sim_plus          1        3.188082          1.0   
5       total_minus          1        0.265382          1.0   
4        total_plus          1        0.670755          1.0   

   mean_interval_score  
3             4.213432  
2            13.530822  
1            12.511540  
0            13.293915  
5            16.365568  
4            26.022942  


In [16]:
results["rel_error_rank"] = (
    results.groupby("d_type")["mean_rel_error"]
    .rank(method="min", ascending=True)
)

results["interval_score_rank"] = (
    results.groupby("d_type")["mean_interval_score"]
    .rank(method="min", ascending=True)
)

results["coverage_rank"] = (
    results.groupby("d_type")["coverage_distance"]
    .rank(method="min", ascending=True)
)

results["overall_rank"] = (
    results["rel_error_rank"]
    + results["interval_score_rank"]
    + results["coverage_rank"]
)

In [17]:
best_overall = (
    results.loc[
        results.groupby("d_type")["overall_rank"].idxmin()
    ]
)

print(best_overall[
    [
        "d_type",
        "model_type",
        "mean_rel_error",
        "pi_coverage",
        "mean_interval_score",
        "overall_rank",
    ]
])


df_to_plot = best_overall[
    [
        "d_type",
        "model_type",
        "mean_rel_error",
        "pi_coverage",
        "mean_interval_score",
        "overall_rank",
    ]
]

fig, ax = table(df_to_plot)
save_path = f"../../{folder}/plots/best_overall.jpg"
fig.savefig(save_path, dpi=300, bbox_inches="tight")
plt.close(fig)

              d_type model_type  mean_rel_error  pi_coverage  \
9   model_form_minus          2        0.037274     0.714286   
2    model_form_plus          1      106.994761     1.000000   
7          sim_minus          2        3.009691     1.000000   
6           sim_plus          2        2.640372     1.000000   
11       total_minus          2        0.137900     0.857143   
10        total_plus          2        0.383520     1.000000   

    mean_interval_score  overall_rank  
9              2.840257           5.0  
2             13.530822           9.0  
7              8.970338           5.0  
6              9.407707           5.0  
11             9.791029           5.0  
10            20.136148           7.0  


In [18]:
metrics_to_plot = {
    "mean_rel_error",
    "pi_coverage",
    "mean_interval_score",
}

for model_type in results["model_type"].unique():

    kernel_results = results[
        results["model_type"] == model_type
    ].copy()

    fig, axes = performance_dist(kernel_results, metrics_to_plot, model_type)
    save_path = f"../../{folder}/plots/perform_dist_{model_type}.jpg"
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)